# 实验二：核心数据结构 "Symbol Sense" 的组装 (Symbol Sense Assembly)

## 🔍 Chapter 6 的真实挑战

教材里充满了同符号多义的"坑"。仔细看 Chapter 6：

- 在 **Table 6.1** 中，$z$ 被定义为："the value of an arbitrary trait"（任意性状的表型值）
- 到了 **6.010 节 (Equation 6.20a)**，作者突然说："Letting $z_i = A_i$ denote the breeding value..."（让 $z_i$ 代表育种值）

同一个字母 $z$，在同一章里含义变了！

## 🧪 实验目的

验证你的代码会不会粗暴地把后一个 $z$ 覆盖掉前一个 $z$。
你要测试 `symbol -> [sense_1, sense_2]` 这种**一对多字典**的写入逻辑。


In [3]:
# ==========================================
# 1. 全局符号-义项字典 (Symbol → Sense List)
# ==========================================

# 这是整个 Knowstellation 的核心数据结构
# key: 符号名 (如 "z", "\sigma", "A")
# value: 义项列表，每个义项是一个 dict
symbol_sense_dict = {}


In [4]:
# ==========================================
# 2. 核心写入函数：add_to_dict
# ==========================================
# 这是你要重点测试的逻辑 ——
# 遇到同名符号时，是"追加"还是"覆盖"？

def add_to_dict(symbol, definition, source):
    """
    向全局符号字典添加一个义项。
    关键逻辑：如果 symbol 已存在，追加新 sense 而不是覆盖！
    """
    safe_key = symbol.replace("\","").replace("{","").replace("}","")
    existing_count = len(symbol_sense_dict.get(symbol, []))
    sense_id = f"chapter6_{safe_key}_{existing_count + 1:03d}"

    new_sense = {
        "sense_id": sense_id,
        "meaning": definition,
        "source": source
    }

    if symbol not in symbol_sense_dict:
        symbol_sense_dict[symbol] = []
    symbol_sense_dict[symbol].append(new_sense)

    print(f"[OK] 已录入: {symbol} → {sense_id} ({definition})")


SyntaxError: unterminated string literal (detected at line 12) (754202993.py, line 12)

In [ ]:
# ==========================================
# 3. 模拟 Chapter 6 的两次读取
# ==========================================

# 清空字典，确保实验干净
symbol_sense_dict.clear()

print("=== 模拟 Chapter 6 的同符号多义读取 ===
")

# 模拟系统第一次读到 Table 6.1 的 z
add_to_dict(symbol="z", definition="arbitrary trait", source="Table 6.1")

# 模拟系统读到 6.010 节的 z
add_to_dict(symbol="z", definition="breeding value", source="6.010")

print("
📊 全局字典当前状态：")


In [ ]:
# ==========================================
# 4. 验证输出：打印全局字典
# ==========================================
import json

print(json.dumps(symbol_sense_dict, indent=2, ensure_ascii=False))


In [ ]:
# ==========================================
# 5. 自动化断言：确保没有发生覆盖灾难
# ==========================================

z_senses = symbol_sense_dict.get("z", [])

print("=== 自动化验证 ===
")

assert len(z_senses) == 2, f"❌ 覆盖灾难！z 应该有 2 个义项，实际只有 {len(z_senses)} 个"
print(f"✅ 断言 1 通过: z 的义项数量 = {len(z_senses)}")

meanings = {s["meaning"] for s in z_senses}
assert "arbitrary trait" in meanings, "❌ 缺少 Table 6.1 的义项！"
assert "breeding value" in meanings, "❌ 缺少 6.010 的义项！"
print(f"✅ 断言 2 通过: 两个含义都存在 → {meanings}")

sense_ids = {s["sense_id"] for s in z_senses}
assert len(sense_ids) == 2, f"❌ sense_id 重复！{sense_ids}"
print(f"✅ 断言 3 通过: sense_id 互不相同 → {sense_ids}")

sources = {s["source"] for s in z_senses}
assert "Table 6.1" in sources, "❌ 缺少 Table 6.1 的来源信息！"
assert "6.010" in sources, "❌ 缺少 6.010 的来源信息！"
print(f"✅ 断言 4 通过: 两个来源都存在 → {sources}")

print("
🎉 全部断言通过！Symbol Sense 组装逻辑正确，不会发生覆盖灾难。")


## 📋 实验结论

| 场景 | 错误做法 | 正确做法 |
|------|----------|----------|
| 遇到同名符号 | `dict[key] = [new_val]` 直接覆盖 | 先 `if key not in dict: dict[key] = []` 再 `append` |
| 数据结构 | `symbol → sense` (一对一) | `symbol → [sense_1, sense_2, ...]` (一对多) |

如果打印出来发现字典里只有 `breeding value`，说明你的代码发生了"覆盖灾难"，后续连图谱会全错。


In [ ]:
# ==========================================
# 6. 进阶测试：更多"同符号多义"场景
# ==========================================

symbol_sense_dict.clear()

print("=== 进阶压力测试：批量同符号多义录入 ===
")

add_to_dict(symbol="\sigma", definition="phenotypic variance", source="Table 6.1")
add_to_dict(symbol="\sigma", definition="standard deviation", source="6.003")
add_to_dict(symbol="\sigma", definition="selection differential", source="6.012")

print()

add_to_dict(symbol="A", definition="breeding value", source="6.010")
add_to_dict(symbol="A", definition="additive genetic variance matrix", source="6.015")

print("
📊 最终字典：")
print(json.dumps(symbol_sense_dict, indent=2, ensure_ascii=False))

print("
=== 验证 ===")
print(f"\sigma 的义项数: {len(symbol_sense_dict.get(\"\sigma\", []))} (期望 3)")
print(f"A 的义项数: {len(symbol_sense_dict.get("A", []))} (期望 2)")

assert len(symbol_sense_dict["\sigma"]) == 3, "\sigma 的义项丢失！"
assert len(symbol_sense_dict["A"]) == 2, "A 的义项丢失！"
print("✅ 进阶测试全部通过！")
